# SVD from Scratch: Matrix Factorisation for Recommenders

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/unsupervised/svd_recommender_systems.ipynb)

Almost everything you watch, buy, or listen to arrives ranked by a recommender, and underneath most rankings sits one idea: factorise a matrix of who-liked-what into a small set of latent tastes. The **singular value decomposition (SVD)** is the classical engine, made famous by the Netflix Prize.

This notebook builds an SVD recommender from scratch on the real **Jester** joke dataset (24,983 users rating 100 jokes), tunes the rank, and beats mean baselines. It then shows the same decomposition is the maths behind **PCA** and **latent semantic indexing (LSI)**.

Companion post: [SVD from Scratch: Matrix Factorisation for Recommenders](https://sesen.ai/blog/svd-from-scratch-recommender-systems).

## 1. Load the Jester dataset

The ratings live in a legacy `.xls` file, which `xlrd` 1.2 reads directly. We keep the dense block of 7,200 users who rated all 100 jokes so plain SVD applies cleanly.

In [ ]:
!pip install -q "xlrd==1.2.0"

In [ ]:
import urllib.request, io, zipfile
import numpy as np
import xlrd
import matplotlib.pyplot as plt

# Download and parse the Jester joke-ratings dataset (24,983 users x 100 jokes).
url = "https://goldberg.berkeley.edu/jester-data/jester-data-1.zip"
xls = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).read("jester-data-1.xls")
sheet = xlrd.open_workbook(file_contents=xls).sheet_by_index(0)
raw = np.array([sheet.row_values(r) for r in range(sheet.nrows)], float)

R = raw[:, 1:]                    # drop the ratings-count column; ratings in [-10, 10]
R[R == 99.0] = np.nan            # 99 marks an unrated joke
dense = R[~np.isnan(R).any(1)]   # 7,200 users who rated all 100 jokes
print("full dataset:", raw.shape, "| dense block:", dense.shape)

## 2. The Quick Win: factorise and predict

Centre each user, fill their held-out gaps with their own mean (a neutral zero after centring), and factorise with `np.linalg.svd`. A rank-7 reconstruction predicts held-out ratings better than a strong user+item bias baseline.

In [ ]:
rng = np.random.default_rng(0)
D = dense[rng.choice(len(dense), 2000, replace=False)]   # 2,000-user demo block

# Hold out 10% of the known ratings to score predictions honestly.
test = rng.random(D.shape) < 0.10
train = np.where(test, np.nan, D)
idx = np.where(test)
truth = D[idx]

# Centre each user, fill gaps with their mean, then factorise.
user_mean = np.nanmean(train, axis=1, keepdims=True)
filled = np.where(np.isnan(train), user_mean, train) - user_mean
U, s, Vt = np.linalg.svd(filled, full_matrices=False)

def svd_predict(k):
    recon = (U[:, :k] * s[:k]) @ Vt[:k] + user_mean
    return recon[idx]

rmse = lambda p: np.sqrt(np.mean((p - truth) ** 2))
mae  = lambda p: np.mean(np.abs(p - truth))

# Baseline: mu + user offset + joke offset (no interactions).
mu = np.nanmean(train)
b_u = np.nanmean(train, axis=1) - mu
b_i = np.nanmean(train, axis=0) - mu
bias_pred = mu + b_u[idx[0]] + b_i[idx[1]]

print(f"global mean RMSE   : {rmse(np.full(truth.shape, mu)):.2f}")
print(f"user+item bias RMSE: {rmse(bias_pred):.2f}")
print(f"rank-7 SVD RMSE    : {rmse(svd_predict(7)):.2f}  (MAE {mae(svd_predict(7)):.2f})")

### What the three matrices mean

`np.linalg.svd` factors `A = U @ diag(s) @ Vt`:

- Each **column of `U`** is a latent *taste* direction, scored for every user.
- Each **row of `Vt`** scores every joke on that same taste.
- The **singular values `s`** say how important each taste is across the crowd.

A predicted rating is the dot product of a user's taste vector and a joke's taste vector, scaled by `s`. Truncating to the top `k` keeps only the strongest, most shared tastes.

## 3. The rank knob

The rank `k` is the only hyperparameter. Too small underfits; too large refits noise. Sweep it and score on the held-out set.

In [ ]:
ks = np.arange(1, 61)
rmse_k = [rmse(svd_predict(k)) for k in ks]
best_k = int(ks[np.argmin(rmse_k)])
print("best rank:", best_k, "| RMSE", round(min(rmse_k), 3))

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(ks, rmse_k, color="#c1121f", lw=2.5, label="truncated SVD")
ax.axhline(rmse(np.full(truth.shape, mu)), color="#8d99ae", ls=":", label="global mean")
ax.axhline(rmse(b_i[idx[1]] + mu), color="#2a9d8f", ls="--", label="item mean")
ax.axhline(rmse(bias_pred), color="#2a6f97", ls="-.", label="user+item bias")
ax.scatter([best_k], [min(rmse_k)], color="#c1121f", s=90, zorder=5)
ax.set_xlabel("rank k"); ax.set_ylabel("held-out RMSE (lower is better)")
ax.set_title("Rank is a knob: too few underfits, too many refits noise")
ax.legend(frameon=False); ax.set_xlim(1, 60)
plt.show()

## 4. Signal in the head, noise in the tail

The leading singular directions carry shared structure that generalises; the long tail carries noise. The best predictive rank keeps only a fraction of the total variance.

In [ ]:
energy = np.cumsum(s ** 2) / np.sum(s ** 2)
print(f"rank {best_k} keeps {energy[best_k-1]*100:.0f}% of the variance, yet predicts best")

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.bar(np.arange(1, 51), s[:50], color="#2a6f97", alpha=0.85)
ax.axvline(best_k, color="#c1121f", ls="--")
ax.set_xlabel("singular value index"); ax.set_ylabel("singular value")
ax2 = ax.twinx()
ax2.plot(np.arange(1, 51), energy[:50], color="#1a1a2e", lw=2)
ax2.set_ylabel("cumulative variance explained"); ax2.set_ylim(0, 1.02)
ax.set_title("One dominant taste direction, then a long noisy tail")
plt.show()

## 5. Reconstruction sharpens, then overfits

Rebuild an 80-user block from a few singular directions. A rank-1 rebuild already captures who is generous and which jokes land; more rank sharpens it, then starts refitting noise.

In [ ]:
SLICE = 80
orig = D[:SLICE]
def recon_slice(k):
    return (U[:SLICE, :k] * s[:k]) @ Vt[:k] + user_mean[:SLICE]

fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
panels = [("original", orig)] + [(f"rank {k} (RMSE {rmse_k[k-1]:.2f})", recon_slice(k)) for k in (1, 7, 30)]
for ax, (title, M) in zip(axes, panels):
    ax.imshow(M, aspect="auto", cmap="RdBu_r", vmin=-10, vmax=10)
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 6. PCA is SVD on centred data

Subtract the column means and take the SVD: the right singular vectors are the principal components. On Iris the two projections match to machine precision.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA

X = load_iris().data
Xc = X - X.mean(0)
Up, sp, Vtp = np.linalg.svd(Xc, full_matrices=False)
scores_svd = (Up * sp)[:, :2]
scores_pca = PCA(n_components=2).fit_transform(Xc)
print("max |PCA - SVD| difference:", np.abs(np.abs(scores_svd) - np.abs(scores_pca)).max())

## 7. LSI is SVD on a term-document matrix

Replace users with words and jokes with documents. The SVD of the word-document counts finds latent *topics*: medical words separate from vehicle words with no labels.

In [ ]:
A = np.array([[2,0,8,6,0,3,1],   # doctor
              [1,6,0,1,7,0,1],   # car
              [5,0,7,4,0,5,6],   # nurse
              [7,0,8,5,0,8,5],   # hospital
              [0,10,0,0,7,0,0]], float)  # wheel
terms = ["doctor", "car", "nurse", "hospital", "wheel"]
Ul, sl, Vtl = np.linalg.svd(A, full_matrices=False)
coords = Ul[:, :2] * sl[:2]

fig, ax = plt.subplots(figsize=(7, 5))
colours = ["#c1121f", "#2a6f97", "#c1121f", "#c1121f", "#2a6f97"]
for (x, y), t, c in zip(coords, terms, colours):
    ax.scatter(x, y, s=140, color=c); ax.annotate(t, (x, y), xytext=(8, 6),
        textcoords="offset points", fontsize=12, fontweight="bold", color=c)
ax.axhline(0, color="#ccc"); ax.axvline(0, color="#ccc")
ax.set_xlabel("latent dimension 1"); ax.set_ylabel("latent dimension 2")
ax.set_title("LSI: SVD separates topics with no labels")
plt.show()

## Exercises

1. **Different rank per metric.** The best rank for RMSE may differ from the best for MAE. Sweep both and compare. Which is more forgiving of large single-rating errors?
2. **Item-item similarity.** Rows of `diag(s) @ Vt` place jokes in taste space. Compute cosine similarity between joke vectors and find the nearest neighbours of joke 0. Do the "similar" jokes make sense?
3. **Top-N recommendations.** For one held-out user, rank their unrated jokes by predicted rating and return the top 5. How does the list change as you vary `k`?
4. **Sparse reality.** Randomly drop 80% of the dense block to mimic a sparse matrix, re-run the mean-fill SVD, and watch the held-out RMSE. Why does the advantage shrink? (See the collaborative filtering post for the gradient-descent fix.)
5. **Eckart-Young check.** Verify numerically that the squared Frobenius error of the rank-`k` reconstruction equals the sum of the squared discarded singular values.

## Further reading

- Eckart & Young (1936), "The approximation of one matrix by another of lower rank".
- Sarwar et al. (2000), "Application of Dimensionality Reduction in Recommender System".
- Deerwester et al. (1990), "Indexing by Latent Semantic Analysis".
- Koren, Bell & Volinsky (2009), "Matrix Factorization Techniques for Recommender Systems".